In [1]:
import os
import pandas as pd

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

C:\Users\priyanshu\AppData\Local\Temp\ipykernel_6772\2343763059.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [2]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

C:\Users\priyanshu\AppData\Local\Temp\ipykernel_6772\2765928982.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [3]:
vectorstore = FAISS.load_local(
    "../vectorstore/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS vectorstore loaded successfully!")

FAISS vectorstore loaded successfully!


In [4]:
query = "Which products are suitable for storage?"

results = vectorstore.similarity_search(
    query,
    k=5
)

print(f"Found {len(results)} results\n")

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)
    print()

Found 5 results

--- Result 1 ---
Product Name: BESTA. Category: Storage. Price: 27499. Description: Durable furniture with practical storage space.. Designer: Jon Karlsson. Dimensions: 69cm width x 33cm depth x 85cm height.
Metadata: {'item_id': 'IKEA00587', 'name': 'BESTA', 'category': 'Storage', 'price': '27499'}

--- Result 2 ---
Product Name: BESTA. Category: Storage. Price: 45933. Description: Compact furniture suitable for small apartments.. Designer: Henrik Preutz. Dimensions: 99cm width x 40cm depth x 43cm height.
Metadata: {'item_id': 'IKEA00584', 'name': 'BESTA', 'category': 'Storage', 'price': '45933'}

--- Result 3 ---
Product Name: IVAR. Category: Storage. Price: 29994. Description: Durable furniture with practical storage space.. Designer: Jon Karlsson. Dimensions: 154cm width x 55cm depth x 163cm height.
Metadata: {'item_id': 'IKEA00044', 'name': 'IVAR', 'category': 'Storage', 'price': '29994'}

--- Result 4 ---
Product Name: IVAR. Category: Storage. Price: 2781. Descri

In [5]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    print("GOOGLE_API_KEY nahi mila. .env file check karo.")
else:
    print("Google API Key loaded successfully!")

Google API Key loaded successfully!


In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

if api_key:
    print("Google API Key loaded successfully!")
else:
    print("GOOGLE_API_KEY nahi mila.")

Google API Key loaded successfully!


In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=api_key
)

print("NEW GEMINI MODEL LOADED")

NEW GEMINI MODEL LOADED


In [8]:
question = "Which storage products are available and what are their prices?"

docs = vectorstore.similarity_search(question, k=4)

context = "\n\n".join([
    f"Product: {doc.metadata.get('name')}\n"
    f"Category: {doc.metadata.get('category')}\n"
    f"Price: {doc.metadata.get('price')} SR\n"
    f"Details: {doc.page_content}"
    for doc in docs
])

prompt = f"""
You are an IKEA product assistant.

Use ONLY the product information provided below.

Product Information:
{context}

User Question:
{question}

Give a clear answer with product names and prices.
"""

response = llm.invoke(prompt)

print("Question:", question)
print("\nAnswer:")
print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Question: Which storage products are available and what are their prices?

Answer:
[{'type': 'text', 'text': 'Based on the provided information, the available storage products and their prices are:\n\n* **BESTA**: 27499 SR\n* **ALEX**: 48727 SR\n* **IVAR**: 29994 SR\n* **BESTA**: 45933 SR', 'extras': {'signature': 'EsYSCsMSARFNMg/u2pJ9WpeIoduQEYyeZs8X5nTwAgbtntuyb3KgPeLqExgIb/gAKc+lpChjSRtEoq7zdOpi5qfD7axEWoRcCcTw5H05MokOQLvY5dzAZQ6WESE4C5b3eoAK0fgSyHmGq3327/XsofTY417EB3sUvWHni8TWpwaMluG1B7o+dMm+7RyHcYa8z/kaJKNyBzjJVJmbS+w+2ux/CLsLlSNFOhVrUl89lOW4p0EooeTooUF/S9zAEFsECsaTOJ5kZjs3HSGdMex0oyxCOp125kaBcXLCLWm/P5/uVWV4NyiWvQfpyGgk1oSgYi9OspYddNSnvWnkoYcS/YQsuKF+6ral7+7OsgYdbCwgZWl3iA8MA/jegBav651sUKj/O2tqCrovwjFsqN95nSN/homOTCPp/ZtTlvbvknqAgt7QPiRPXldQ3q3sauys35uOKW0HR2R8shYeqeruj/QHy+I4S6XNwcstgc7nwA1EH5ZMFBTJW2m3ZC7x6SRFZ7HBoH70svrhDGkw2hGnm/D55AnMPn1+VVMiEtxrKs7VPGjobT3UcPObAMjcc1eUjsfUEiSe9TeUcIsWd7a10VKo7r8nTEAxm/yDmazJFaT4NLuTlbeVYbjtFavruz2rYV1VYpQIGynkwP6H/xso+zXf1GggLC0690mwfzNCjqd

In [9]:
user_question = input("Ask your IKEA question: ")

docs = vectorstore.similarity_search(user_question, k=4)

context = "\n\n".join([
    f"Product: {doc.metadata.get('name')}\n"
    f"Category: {doc.metadata.get('category')}\n"
    f"Price: {doc.metadata.get('price')} SR\n"
    f"Details: {doc.page_content}"
    for doc in docs
])

prompt = f"""
You are an IKEA product assistant.

Answer the user's question using ONLY the product information below.

Product Information:
{context}

User Question:
{user_question}

Give a clear and helpful answer.
"""

response = llm.invoke(prompt)

print("\nIKEA Assistant:")
print(response.content)


IKEA Assistant:
[{'type': 'text', 'text': 'Based on our current inventory, here are the storage products available:\n\n1. **HEMNES**\n   * **Price:** 24,871 SR\n   * **Description:** Durable furniture with practical storage space.\n   * **Designer:** IKEA Design Team\n   * **Dimensions:** 102cm width x 38cm depth x 120cm height\n\n2. **BESTA (Option 1)**\n   * **Price:** 27,499 SR\n   * **Description:** Durable furniture with practical storage space.\n   * **Designer:** Jon Karlsson\n   * **Dimensions:** 69cm width x 33cm depth x 85cm height\n\n3. **IVAR**\n   * **Price:** 29,994 SR\n   * **Description:** Durable furniture with practical storage space.\n   * **Designer:** Jon Karlsson\n   * **Dimensions:** 154cm width x 55cm depth x 163cm height\n\n4. **BESTA (Option 2)**\n   * **Price:** 45,933 SR\n   * **Description:** Compact furniture suitable for small apartments.\n   * **Designer:** Henrik Preutz\n   * **Dimensions:** 99cm width x 40cm depth x 43cm height\n\nDepending on your sp